<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/IPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import json
import time
import io
import requests
import pandas as pd

years = [2024, 2025, 2026]

# Mimic a standard browser handshake to prevent 403 forbidden responses
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5"
}

print("Initiating StockAnalysis IPO data retrieval pipeline...")

all_yearly_dfs = [] # List to store DataFrames for each year

for year in years:
    # Target URL pattern for historical yearly listings
    url = f"https://stockanalysis.com/ipos/{year}/"
    print(f"Fetching: {url}")

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)

        if response.status_code == 200:
            # Wrap the raw HTML text in a StringIO block to satisfy newer pandas requirements
            tables = pd.read_html(io.StringIO(response.text))

            if tables:
                # The primary historical data grid is the first table on the page
                df_year = tables[0]
                all_yearly_dfs.append(df_year) # Add the current year's DataFrame to the list

                # Transform the data frame into a structured JSON payload
                raw_payload = {
                    "year": year,
                    "source": "stockanalysis.com",
                    "count": len(df_year),
                    "ipos": df_year.to_dict(orient="records")
                }

                output_filename = f"/content/ipo_state_{year}.json"
                with open(output_filename, "w", encoding="utf-8") as f:
                    json.dump(raw_payload, f, ensure_ascii=False, indent=4)

                print(f"     ✅ Successfully saved {len(df_year)} listings to: {output_filename}")
            else:
                print(f"     ❌ Table element not found in HTML structure for year {year}.")
        else:
            print(f"     ❌ Failed connection to {year}. HTTP Status: {response.status_code}")

    except Exception as e:
        print(f"     ❌ Scraping exception encountered for year {year}: {str(e)}")

    # 2-second decay delay to be a polite scraper and prevent rate limits
    time.sleep(2.0)

# Concatenate all yearly DataFrames into a single DataFrame
if all_yearly_dfs:
    consolidated_df = pd.concat(all_yearly_dfs, ignore_index=True)
    print("\n--- Consolidated DataFrame Head Sample ---")
    display(consolidated_df.head())
    print("\n--- Consolidated DataFrame Tail Sample ---")
    display(consolidated_df.tail())
else:
    print("\nNo data was retrieved to consolidate.")

print("\nRetrieval step complete. Proceed to Cell 2 to process.")

Initiating StockAnalysis IPO data retrieval pipeline...
Fetching: https://stockanalysis.com/ipos/2024/
     ✅ Successfully saved 225 listings to: /content/ipo_state_2024.json
Fetching: https://stockanalysis.com/ipos/2025/
     ✅ Successfully saved 347 listings to: /content/ipo_state_2025.json
Fetching: https://stockanalysis.com/ipos/2026/
     ✅ Successfully saved 199 listings to: /content/ipo_state_2026.json

--- Consolidated DataFrame Head Sample ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$4.00,$1.15,-71.25%
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",-,$0.453,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$4.00,$1.02,-73.50%
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.00,$10.71,7.10%
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.00,$10.67,6.70%



--- Consolidated DataFrame Tail Sample ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return
766,"Jan 8, 2026",BBCQ,Bleichroeder Acquisition Corp. II,$10.00,$10.32,3.30%
767,"Jan 8, 2026",BUDA,"Buda Juice, Inc.",$7.50,$11.02,44.80%
768,"Jan 7, 2026",SORN,Soren Acquisition Corp.,$10.00,$9.94,-0.60%
769,"Jan 6, 2026",ARTC,Art Technology Acquisition Corp.,$10.00,$9.98,-0.20%
770,"Jan 6, 2026",BIII,Black Spade Acquisition III Co,$10.00,$9.94,-0.60%



Retrieval step complete. Proceed to Cell 2 to process.


### Loading and Merging 'IPO Deal Size' Data

Now, let's load the provided `IPO_Deal_Size.csv` file and merge its information (Deal Size) into our `consolidated_df`. We will assume the 'Symbol' column is present in both DataFrames for the merge operation.

In [12]:
# Load the IPO_Deal_Size.txt file as a pipe-separated file, handling quoted fields
ipo_deal_size_df = pd.read_csv('/content/IPO_Deal_Size.txt', sep='|', quotechar='"', skipinitialspace=True)

print("--- IPO Deal Size DataFrame Head ---")
display(ipo_deal_size_df.head())

--- IPO Deal Size DataFrame Head ---


,Symbol,Exchange,IPO Price,Shares Offered,Deal Size
0,RACD,NASDAQ,$10.00,"7,500,000",75.00M
1,SAMO,NYSE,$10.00,"20,000,000",200.00M
2,SKHY,NASDAQ,$149.00,"177,900,000",28.13B
3,CCCT,NASDAQ,$10.00,"20,000,000",200.00M
4,MRCO,NASDAQ,$10.00,"15,000,000",150.00M


In [14]:
# Merge the ipo_deal_size_df with consolidated_df on the 'Symbol' column
# We will use a left merge to keep all original IPOs from consolidated_df
consolidated_df_with_deal_size = pd.merge(
    consolidated_df,
    ipo_deal_size_df[['Symbol', 'Shares Offered', 'Deal Size']],
    on='Symbol',
    how='left'
)

print("--- Consolidated DataFrame with Deal Size (Head) ---")
display(consolidated_df_with_deal_size.head())

print("--- Consolidated DataFrame with Deal Size (Tail) ---")
display(consolidated_df_with_deal_size.tail())

print("\nMerge complete. The `consolidated_df_with_deal_size` DataFrame now includes 'Deal Size' and 'Shares Offered'.")

--- Consolidated DataFrame with Deal Size (Head) ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Shares Offered,Deal Size
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$4.00,$1.15,-71.25%,"1,750,000",7.00M
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",-,$0.453,-,-,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$4.00,$1.02,-73.50%,"2,300,000",9.20M
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.00,$10.71,7.10%,"15,000,000",150.00M
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.00,$10.67,6.70%,"10,000,000",100.00M


--- Consolidated DataFrame with Deal Size (Tail) ---


,IPO Date,Symbol,Company Name,IPO Price,Current,Return,Shares Offered,Deal Size
768,"Jan 8, 2026",BBCQ,Bleichroeder Acquisition Corp. II,$10.00,$10.32,3.30%,"25,000,000",250.00M
769,"Jan 8, 2026",BUDA,"Buda Juice, Inc.",$7.50,$11.02,44.80%,"2,666,667",20.00M
770,"Jan 7, 2026",SORN,Soren Acquisition Corp.,$10.00,$9.94,-0.60%,"22,000,000",220.00M
771,"Jan 6, 2026",ARTC,Art Technology Acquisition Corp.,$10.00,$9.98,-0.20%,"22,000,000",220.00M
772,"Jan 6, 2026",BIII,Black Spade Acquisition III Co,$10.00,$9.94,-0.60%,"15,000,000",150.00M



Merge complete. The `consolidated_df_with_deal_size` DataFrame now includes 'Deal Size' and 'Shares Offered'.
